# Phase 3 — experiments queue (STID, AGCRN and the probabilistic AGCRN on PEMS04 / PEMS08)
Run the cells **one by one** (Shift+Enter). Needs a **GPU runtime** (Runtime → Change runtime type → T4 GPU).
Everything that must survive a disconnect lives on **Google Drive** (checkpoints + one results JSON per finished job).
If the session dies: reopen this notebook, run cells 1-5 again, then the queue cell again — finished jobs are skipped and the interrupted one resumes.

Models: `stid`, `agcrn` (deterministic baselines) and `agcrn_prob` (mean + variance, trained with the Gaussian NLL).
Step A: `RUN_MODE = "smoke"` (1 epoch, tiny AGCRN) to test the whole queue quickly.
Step B: `RUN_MODE = "full"` for the real 100-epoch runs.

## 1. Which hardware do I have?

In [ ]:
!nvidia-smi 2>&1 | head -15
import sys
print(sys.version)

## 2. Mount Google Drive (a pop-up will ask for permission)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get the repo and build the Python 3.11 environment (a few minutes; reused if it already exists)

In [ ]:
import os
%cd /content
if os.path.isdir("/content/proposal/.git"):
    !cd /content/proposal && git pull
else:
    !git clone --recursive https://github.com/hrfpour/proposal.git
%cd /content/proposal/code/BasicTS
if not os.path.exists("/content/venv/bin/python"):
    !pip -q install uv
    !uv venv /content/venv --python 3.11
    !uv pip install --python /content/venv/bin/python -r requirements.txt 2>&1 | tail -n 25
else:
    print("venv already exists - skipping install")

## 4. Make sure PyTorch exists in that environment, and that BasicTS imports

In [ ]:
import subprocess
PY = "/content/venv/bin/python"
if subprocess.run([PY, "-c", "import torch"], capture_output=True).returncode != 0:
    print("torch not in requirements.txt -> installing it")
    !uv pip install --python {PY} torch
!{PY} -c "import sys; sys.path.insert(0, '/content/proposal/code/BasicTS/src'); import basicts, torch, numpy; print('basicts', basicts.__version__, '| torch', torch.__version__, '| numpy', numpy.__version__, '| cuda', torch.cuda.is_available())"

## 5. Put the PEMS04 / PEMS08 data in place (from the PEMS-only copy cached in your Drive)

In [ ]:
import os, shutil
KEEP = ["PEMS04", "PEMS08"]
CACHE = "/content/drive/MyDrive/proposal_data"
BT = "/content/proposal/code/BasicTS"
assert all(os.path.isdir(f"{CACHE}/{n}") for n in KEEP), "PEMS cache missing in Drive (proposal_data): run notebook 00 first"
os.makedirs(f"{BT}/datasets", exist_ok=True)
for n in KEEP:
    shutil.copytree(f"{CACHE}/{n}", f"{BT}/datasets/{n}", dirs_exist_ok=True)
!ls {BT}/datasets

## 5b. Check that our AGCRN code still equals the official one (10 seconds)
Needed after the small dropout change in `agcrn.py`. Expected last line: `RESULT: PASS (port is equivalent to the official code)`.

In [ ]:
import os
if not os.path.isdir("/content/AGCRN_official"):
    !git clone --depth 1 https://github.com/LeiBAI/AGCRN /content/AGCRN_official
!/content/venv/bin/python /content/proposal/code/scripts/check_agcrn_equivalence.py --official /content/AGCRN_official

## 6. Choose what to run
`RUN_MODE = "smoke"` first. When it works, change to `"full"`.

Presets (set the variables in the next cell):
- **deterministic baselines:** `MODELS = ["stid", "agcrn"]`
- **probabilistic AGCRN (aleatoric):** `MODELS = ["agcrn_prob"]`, `DROPOUT = 0.0`, `MC_SAMPLES = 0`, `TAG = ""`
- **probabilistic AGCRN + MC Dropout (aleatoric + epistemic):** `MODELS = ["agcrn_prob"]`, `DROPOUT = 0.1`, `MC_SAMPLES = 20` (smoke: `5`), `TAG = "drop10"`

Results of different presets never mix: the tag becomes part of the model name (`AGCRN_PROB_DROP10`).
To split the work over several sessions, shorten `MODELS` / `DATASETS` / `SEEDS`.

In [ ]:
RUN_MODE = "smoke"          # "smoke" = 1 epoch pipeline test | "full" = real 100-epoch baselines
MODELS   = ["stid", "agcrn"]
DATASETS = ["PEMS04", "PEMS08"]
SEEDS    = [42]             # later: [42, 43, 44]
GPUS     = "0"              # "none" = CPU (smoke mode only)
DROPOUT    = 0.0            # agcrn / agcrn_prob: dropout between layers (0 = off)
MC_SAMPLES = 0              # agcrn_prob: MC-Dropout passes in the final evaluation (smoke: 5, full: 20)
TAG        = ""             # model-name suffix, e.g. "drop10"

if RUN_MODE == "smoke":
    EPOCHS, STATUS, TINY = 1, "smoke_test", True
    CKPT_ROOT   = "/content/drive/MyDrive/proposal_ckpt_smoke"
    RESULTS_DIR = "/content/drive/MyDrive/proposal_runs/queue_smoke"
else:
    EPOCHS, STATUS, TINY = 100, "done", False
    CKPT_ROOT   = "/content/drive/MyDrive/proposal_ckpt"
    RESULTS_DIR = "/content/drive/MyDrive/proposal_runs/full"

PY = "/content/venv/bin/python"
ARGS = (f"--models {' '.join(MODELS)} --datasets {' '.join(DATASETS)} --seeds {' '.join(map(str, SEEDS))} "
        f"--epochs {EPOCHS} --gpus {GPUS} --ckpt-root {CKPT_ROOT} --results-dir {RESULTS_DIR} --status {STATUS}"
        + (" --tiny" if TINY else "")
        + (f" --dropout {DROPOUT}" if DROPOUT > 0 else "")
        + (f" --mc-samples {MC_SAMPLES}" if MC_SAMPLES > 1 else "")
        + (f" --tag {TAG}" if TAG else ""))
print(ARGS)
!{PY} /content/proposal/code/scripts/run_queue.py {ARGS} --dry-run

## 7. Run the queue
Lines you will see: `[i/N] RUN ...`, `Epoch k / E`, `Result <val>/<test>`, `DONE ... in X min`, and at the end `Finished N / N`.
Do not close the tab while it runs. If the session disconnects, run cells 1-6 and this cell again.

In [ ]:
!{PY} /content/proposal/code/scripts/run_queue.py {ARGS}

## 8. Summary tables of the finished jobs (MAPE in %)

In [ ]:
import json, glob
rows, uq_rows = [], []
for f in sorted(glob.glob(f"{RESULTS_DIR}/*.json")):
    if f.endswith("_cfg.json") or f.endswith("_test_metrics.json"):
        continue
    d = json.load(open(f)); pm = d["point_metrics"]
    rows.append((d["model"], d["dataset"], d["seed"], d["status"], d.get("epochs_trained"),
                 pm["mae"], pm["rmse"], pm["mape"] * 100, (d.get("train_time_s") or 0) / 60, d.get("num_parameters")))
    u = d.get("uq_metrics") or {}
    if u.get("picp") is not None:
        uq_rows.append((d["model"], d["dataset"], d["seed"], u["picp_90"], u["picp_95"], u["mpiw_95"], u["nll"], u["crps"], u["ece"]))
print(f"{'model':11}{'dataset':8}{'seed':5}{'status':11}{'epochs':7}{'MAE':>8}{'RMSE':>8}{'MAPE%':>8}{'min':>7}{'params':>9}")
for r in rows:
    print(f"{r[0]:11}{r[1]:8}{r[2]:<5}{r[3]:11}{str(r[4]):7}{r[5]:8.2f}{r[6]:8.2f}{r[7]:8.2f}{r[8]:7.1f}{str(r[9]):>9}")
if uq_rows:
    print(f"\nUncertainty (original unit; nominal PICP90 = 0.90, PICP95 = 0.95)")
    print(f"{'model':11}{'dataset':8}{'seed':5}{'PICP90':>8}{'PICP95':>8}{'MPIW95':>9}{'NLL':>8}{'CRPS':>8}{'ECE':>8}")
    for r in uq_rows:
        print(f"{r[0]:11}{r[1]:8}{r[2]:<5}{r[3]:8.3f}{r[4]:8.3f}{r[5]:9.2f}{r[6]:8.3f}{r[7]:8.3f}{r[8]:8.3f}")
print("\nFor reference (paper-reported, NOT reproduced): AGCRN PEMS04 MAE 19.83 RMSE 32.26 MAPE 12.97 | AGCRN PEMS08 MAE 15.95 RMSE 25.22 MAPE 10.09")

## 9. (probabilistic runs) Was the predicted variance saved? Expect one `log_var.npy` per agcrn_prob run

In [ ]:
!find {CKPT_ROOT} -name log_var.npy 2>/dev/null | head